# Lab Day 1 — Mạng nơ-ron và huấn luyện

**Sinh viên:** VuongVietHoang — **MSSV:** 2A202602528

Tải nguyên thư mục `submission_2A202602528` lên Google Drive, giữ đủ `code/*.py`, rồi mở `submission_2A202602528/code/lab.ipynb` bằng Google Colab. Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chọn **Runtime → Run all**. Notebook dùng các module đã hoàn thiện trong thư mục Drive, clone repo để lấy dữ liệu/script gốc và lưu kết quả trong cùng thư mục submission trên Drive.

Mỗi thí nghiệm chỉ thay một yếu tố. Cấu hình được chọn bằng validation; tập `eval` chỉ được chấm sau khi đã chốt baseline và cấu hình cuối. Mỗi thí nghiệm lưu JSON và ảnh ngay sau khi chạy để hạn chế mất kết quả nếu Colab ngắt.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, math, time
import numpy as np
import torch

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/VietHoang04-sys/K4-Track4-Day1-VuongVietHoang-2A202602528-Neuralnetwork.git"
REPO_ROOT = Path("/content/K4-Track4-Day1-VuongVietHoang-2A202602528-Neuralnetwork")
OUT_DIR = Path("/content/drive/MyDrive/submission_2A202602528")
CODE_DIR = OUT_DIR / "code"
module_names = ("data.py", "model.py", "optimizer.py", "train.py", "plots.py", "results_table.py")
missing_modules = [name for name in module_names if not (CODE_DIR / name).is_file()]
if missing_modules:
    raise FileNotFoundError(
        f"Upload the complete submission_2A202602528 folder to Google Drive first; missing: {missing_modules}"
    )
if not (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
for directory in ("figures", "results"):
    (OUT_DIR / directory).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))
os.chdir(REPO_ROOT)

print("Repository:", REPO_ROOT)
print("Submission folder on Drive:", OUT_DIR)
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slower)")


## Part 0 — Chuẩn bị dữ liệu

Chạy script chia tập theo metadata có sẵn. Validation được tách phân tầng từ train với seed 42. Chỉ 10 đặc trưng số đầu tiên được chuẩn hoá theo thống kê train; không sửa metadata và không dùng eval để chuẩn hoá.


In [ ]:
subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)
from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, activation_stats
from optimizer import build_optimizer
from train import DEFAULT_CFG, evaluate, run_experiment, final_eval, set_seed
from plots import plot_run, plot_compare
from results_table import save_result, to_row, write_xlsx

device = "cuda" if torch.cuda.is_available() else "cpu"
data = prepare_data(device=device, val_fraction=0.2, seed=42,
                    processed_dir=str(REPO_ROOT / "data" / "processed"))


## Part 1 — Model và kiểm tra sức khoẻ

Kiến trúc bắt buộc M-base: `54 → 256 → 128 → 7`, 47.879 tham số. Ghi loss bước 0, kiểm tra gradient và thử overfit 20 mẫu trước khi chạy huấn luyện dài.


In [ ]:
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47879
logits = model(torch.zeros(8, 54, device=device))
assert logits.shape == (8, 7)
print("Logits:", tuple(logits.shape), "parameters:", count_params(model))
print("Activation std after hidden ReLUs on validation batch:",
      activation_stats(model, data["X_val"][:512]))
print("Initial validation CE:", evaluate(model, data["X_val"], data["y_val"])["loss"],
      "(uniform-logit reference ln(7)=", math.log(7), ")")

# Gradient-flow check on a real batch.
model.train()
model.zero_grad(set_to_none=True)
loss = torch.nn.functional.cross_entropy(model(data["X_tr"][:64]), data["y_tr"][:64])
loss.backward()
grad_report = {name: (parameter.grad is not None,
                       float(parameter.grad.norm().item()) if parameter.grad is not None else None)
               for name, parameter in model.named_parameters()}
print("Gradient norms:", grad_report)
assert all(present and norm > 0 for present, norm in grad_report.values())

# Tiny-batch memorization sanity check (20 examples; regularization off).
set_seed(7)
tiny_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
X_tiny, y_tiny = data["X_tr"][:20], data["y_tr"][:20]
for step in range(400):
    tiny_opt.zero_grad(set_to_none=True)
    tiny_loss = torch.nn.functional.cross_entropy(tiny_model(X_tiny), y_tiny)
    tiny_loss.backward()
    tiny_opt.step()
tiny_model.eval()
with torch.no_grad():
    tiny_final_loss = torch.nn.functional.cross_entropy(tiny_model(X_tiny), y_tiny).item()
    tiny_accuracy = (tiny_model(X_tiny).argmax(1) == y_tiny).float().mean().item()
print(f"20-sample overfit check: loss={tiny_final_loss:.6f}, accuracy={tiny_accuracy:.3f}")


## Part 2 — Chọn learning rate và baseline

Chọn learning rate bằng validation: sàng lọc ba mức SGD+momentum và hai mức Adam, mỗi mức 4 epoch. Các pilot chỉ dùng chọn learning rate, không so sánh như kết quả cuối với thí nghiệm 20 epoch. Sau đó chạy baseline đầy đủ 20 epoch cho seed 1 và seed 2.


In [ ]:
all_results = []
result_by_id = {}

def run_and_record(cfg):
    result = run_experiment(cfg, data)
    save_result(result, str(OUT_DIR / "results"))
    plot_run(result, str(OUT_DIR / "figures" / f"{cfg['exp_id']}.png"))
    all_results.append(result)
    result_by_id[cfg["exp_id"]] = result
    return result

base = {**DEFAULT_CFG, "exp_id": "base-s1", "group": "baseline",
        "description": "M-base; CE; SGD momentum 0.9; He",
        "loss": "ce", "optimizer": "sgd_momentum", "weight_decay": 0.0,
        "momentum": 0.9, "batch": 512, "epochs": 20, "hidden": (256, 128),
        "dropout": 0.0, "clip_norm": None, "precision": "fp32",
        "init": "he", "seed": 1}

sgd_pilots = []
for lr in (0.01, 0.03, 0.06):
    cfg = {**base, "exp_id": f"lr-sgd-{lr:g}", "group": "hparam",
           "description": f"SGD momentum learning-rate pilot lr={lr}; 4 epochs",
           "lr": lr, "epochs": 4}
    sgd_pilots.append(run_and_record(cfg))
best_sgd_pilot = max(sgd_pilots, key=lambda r: r["summary"]["val_macro_f1"])
selected_sgd_lr = best_sgd_pilot["cfg"]["lr"]
print("Selected SGD learning rate from validation pilots:", selected_sgd_lr)

adam_pilots = []
for lr in (0.001, 0.003):
    cfg = {**base, "exp_id": f"lr-adam-{lr:g}", "group": "optimizer",
           "description": f"Adam learning-rate pilot lr={lr}; 4 epochs",
           "optimizer": "adam", "lr": lr, "epochs": 4}
    adam_pilots.append(run_and_record(cfg))
best_adam_pilot = max(adam_pilots, key=lambda r: r["summary"]["val_macro_f1"])
selected_adam_lr = best_adam_pilot["cfg"]["lr"]
print("Selected Adam learning rate from validation pilots:", selected_adam_lr)

base["lr"] = selected_sgd_lr
baseline = run_and_record(base)
baseline_seed2 = run_and_record({**base, "exp_id": "base-s2", "seed": 2,
                                  "description": "M-base baseline, seed 2"})


## Part 3 — Các chủ đề thí nghiệm

Giả thuyết ghi trước khi chạy: CE có thể phù hợp hơn MSE cho phân loại; Adam có thể giảm loss sớm nhanh hơn khi đã chọn learning rate; M-wide có thể tăng năng lực nhưng tốn thời gian; dropout hữu ích khi có overfit; clipping chỉ tác động khi norm vượt ngưỡng; FP16 có thể giảm thời gian/bộ nhớ trên GPU phù hợp nhưng cần GradScaler; He và Xavier có thể cho độ lớn kích hoạt khác nhau. Đây là dự đoán, không phải kết luận.

Mỗi thí nghiệm chính giữ seed 1 và 20 epoch, chỉ đổi một yếu tố so với baseline. Bỏ qua BF16 nếu GPU không hỗ trợ.


In [ ]:
experiment_cfgs = [
    {**base, "exp_id": "loss-mse", "group": "loss",
     "description": "MSE loss vs baseline CE", "loss": "mse"},
    {**base, "exp_id": "opt-adam", "group": "optimizer",
     "description": "Adam; learning rate selected with validation pilot",
     "optimizer": "adam", "lr": selected_adam_lr},
    {**base, "exp_id": "hparam-wide", "group": "hparam",
     "description": "M-wide (512-256) vs M-base", "hidden": (512, 256)},
    {**base, "exp_id": "dropout-03", "group": "dropout",
     "description": "Dropout p=0.3 after hidden ReLUs", "dropout": 0.3},
    {**base, "exp_id": "clip-1", "group": "clipping",
     "description": "Global gradient clipping at norm 1.0", "clip_norm": 1.0},
    {**base, "exp_id": "init-xavier", "group": "init",
     "description": "Xavier-normal initialization vs He", "init": "xavier"},
]
if torch.cuda.is_available():
    experiment_cfgs.append({**base, "exp_id": "amp-fp16", "group": "amp",
                            "description": "FP16 autocast + GradScaler", "precision": "fp16"})
else:
    print("Skipping FP16 experiment: this runtime has no CUDA GPU.")

for cfg in experiment_cfgs:
    run_and_record(cfg)


In [ ]:
# Save direct comparisons for experiment groups that have multiple runs.
for group in sorted({result["cfg"]["group"] for result in all_results}):
    group_results = [r for r in all_results if r["cfg"]["group"] == group]
    if len(group_results) > 1:
        plot_compare(group_results, "val_macro_f1",
                     str(OUT_DIR / "figures" / f"compare_{group}.png"),
                     title=f"Validation macro-F1: {group}")


### Ghi nhận và phân tích

Mở `results/<exp_id>.json` và `figures/<exp_id>.png` trên Drive. So sánh với baseline seed 1 và ngưỡng nhiễu `2σ` từ hai seed baseline. Không so trực tiếp độ lớn CE loss với MSE loss. Grad norm được đo trước clipping; xem đường cong để đánh giá clipping có cơ hội kích hoạt hay không.


## Part 4 — Chọn bằng validation, chấm eval, xuất sản phẩm

Cấu hình cuối được chọn theo validation macro-F1 trong các thí nghiệm chính seed 1 (không gồm learning-rate pilot). Checkpoint tốt nhất của mỗi lần chạy được chọn theo val loss. Sau khi chốt, script chính thức chấm eval cho baseline và cấu hình cuối; không dùng eval để thay đổi lựa chọn.


In [ ]:
main_results = [r for r in all_results
                if r["cfg"]["seed"] == 1 and r["cfg"]["exp_id"].startswith((
                    "base-s1", "loss-", "opt-", "hparam-wide", "dropout-", "clip-", "amp-", "init-"))]
final_result = max(main_results, key=lambda r: r["summary"]["val_macro_f1"])
final_id = final_result["cfg"]["exp_id"]
print("Final configuration selected on validation:", final_id,
      "val macro-F1=", final_result["summary"]["val_macro_f1"])

def run_official_evaluation(pred_path, json_path):
    completed = subprocess.run(
        [sys.executable, "scripts/evaluate.py", "--pred", str(pred_path),
         "--out", str(json_path)], cwd=REPO_ROOT, check=True,
        capture_output=True, text=True)
    print(completed.stdout)
    with open(json_path, encoding="utf-8") as stream:
        return json.load(stream)

baseline_csv = OUT_DIR / "results" / "predictions_baseline.csv"
baseline_json = OUT_DIR / "results" / "eval_baseline.json"
final_eval(base, baseline, data, str(baseline_csv))
baseline_eval_scores = run_official_evaluation(baseline_csv, baseline_json)

if final_id == "base-s1":
    shutil.copy2(baseline_csv, OUT_DIR / "predictions_eval.csv")
    final_eval_scores = baseline_eval_scores
else:
    final_eval(final_result["cfg"], final_result, data, str(OUT_DIR / "predictions_eval.csv"))
    final_eval_scores = run_official_evaluation(
        OUT_DIR / "predictions_eval.csv", OUT_DIR / "eval_result.json")
if final_id == "base-s1":
    with open(OUT_DIR / "eval_result.json", "w", encoding="utf-8") as stream:
        json.dump(final_eval_scores, stream, ensure_ascii=False, indent=2)

eval_by_exp = {"base-s1": baseline_eval_scores, final_id: final_eval_scores}
rows = [to_row(r, eval_scores=eval_by_exp.get(r["cfg"]["exp_id"]))
        for r in all_results]
write_xlsx(rows, str(REPO_ROOT / "templates" / "experiment_table_template.xlsx"),
           str(OUT_DIR / "experiments.xlsx"))
print("Submission outputs written to", OUT_DIR)


## Phân tích eval và báo cáo

Notebook tạo bảng, biểu đồ, `predictions_eval.csv`, `eval_result.json` và báo cáo từ số liệu đo được. Kiểm tra F1 từng lớp và ma trận nhầm lẫn; bổ sung phản tư cá nhân nếu kết quả khác dự đoán.


In [ ]:
def write_measured_report():
    base_runs = [r for r in all_results if r["cfg"]["group"] == "baseline"]
    seed_f1 = np.asarray([r["summary"]["val_macro_f1"] for r in base_runs], dtype=float)
    seed_acc = np.asarray([r["summary"]["val_acc"] for r in base_runs], dtype=float)
    sigma_f1 = float(seed_f1.std(ddof=1)) if len(seed_f1) > 1 else float("nan")
    baseline_f1 = baseline["summary"]["val_macro_f1"]
    lines = [
        "# Báo cáo Lab Day 1 — VuongVietHoang — 2A202602528", "",
        "## 1. Thiết lập",
        f"- Môi trường: Google Colab; PyTorch {torch.__version__}; thiết bị: " +
        (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU") + ".",
        "- Chia train/eval cố định; validation phân tầng 20% train, seed 42. "
        "Chuẩn hoá 10 cột số theo thống kê train; giữ nguyên 44 cột nhị phân.",
        f"- M-base có {EXPECTED_PARAMS[(256, 128)]:,} tham số; baseline dùng CE, "
        f"SGD+momentum, lr={base['lr']}, batch=512, 20 epoch, He.",
        f"- Validation majority-class accuracy: {data['majority_val_accuracy']:.4f}.", "",
        "## 2. Kiểm tra ban đầu và độ nhiễu",
        f"- Loss bước 0 (baseline): {baseline['summary']['step0_loss']:.5f}; "
        f"tham chiếu ln(7)={math.log(7):.5f}.",
        f"- Overfit 20 mẫu: loss={tiny_final_loss:.6f}, accuracy={tiny_accuracy:.3f}.",
        f"- Gradient khác 0 ở mọi tham số: {all(present and norm > 0 for present, norm in grad_report.values())}.",
        f"- Baseline validation accuracy trung bình ± SD ({len(seed_acc)} seed): "
        f"{seed_acc.mean():.4f} ± {seed_acc.std(ddof=1) if len(seed_acc)>1 else 0:.4f}.",
        f"- Baseline validation macro-F1 trung bình ± SD ({len(seed_f1)} seed): "
        f"{seed_f1.mean():.4f} ± {sigma_f1 if len(seed_f1)>1 else 0:.4f}; "
        f"2σ={2*sigma_f1 if len(seed_f1)>1 else 'chưa ước lượng được'}.", "",
        "## 3. Kết quả validation theo thí nghiệm",
        "LR pilot chạy 4 epoch chỉ dùng chọn learning rate; không xem là so sánh hội tụ tương đương với thí nghiệm chính 20 epoch.",
        "| exp_id | nhóm | thay đổi | best epoch | val accuracy | val macro-F1 | giây/epoch | biểu đồ |",
        "|---|---|---|---:|---:|---:|---:|---|",
    ]
    for r in all_results:
        c, s = r["cfg"], r["summary"]
        lines.append(f"| {c['exp_id']} | {c['group']} | {c['description']} | {s['best_epoch']} | "
                     f"{s['val_acc']:.4f} | {s['val_macro_f1']:.4f} | {s['time_per_epoch_s']:.2f} | "
                     f"[ảnh](figures/{c['exp_id']}.png) |")

    explanations = {
        "loss": ("Dự đoán: CE có thể phân loại tốt hơn MSE vì gradient CE phản ánh trực tiếp sai số xác suất lớp.",
                 "Giải thích: CE tối ưu từ logits với softmax-cross-entropy; MSE tối ưu khoảng cách với one-hot. So macro-F1/accuracy, không so độ lớn hai loss."),
        "optimizer": ("Dự đoán: Adam có thể giảm loss nhanh hơn nhờ bước cập nhật thích nghi theo từng tham số.",
                      "Giải thích: SGD+momentum tích luỹ hướng gradient; Adam chuẩn hoá theo moment. Các learning rate được sàng lọc trên validation, nhưng pilot ngắn nên đây chưa phải so sánh hội tụ hoàn toàn."),
        "hparam": ("Dự đoán: learning rate tốt hơn có thể tăng tốc hội tụ; M-wide có thể biểu diễn ranh giới phức tạp hơn nhưng tăng chi phí.",
                   "Giải thích: đối chiếu val macro-F1, best epoch và giây/epoch; số bước cập nhật mỗi epoch giữ nguyên với batch 512."),
        "dropout": ("Dự đoán: dropout chỉ hữu ích nếu baseline có dấu hiệu overfit.",
                    "Giải thích: đối chiếu khoảng cách train-val loss và val macro-F1; dropout giảm đồng thích nghi nhưng cũng làm khó tối ưu hơn."),
        "clipping": ("Dự đoán: clipping norm 1.0 chỉ ảnh hưởng nếu grad norm trước clip đủ lớn.",
                     "Giải thích: đường ghi nhận là norm trung bình trước clip theo epoch; nếu giá trị thấp, cần thận trọng khi kết luận clipping đã có tác dụng."),
        "amp": ("Dự đoán: FP16 có thể giảm bộ nhớ/thời gian trên GPU tương thích, nhưng chất lượng dự đoán gần FP32.",
                "Giải thích: dựa trên số đo thời gian, peak memory và validation; FP16 dùng GradScaler để hạn chế underflow."),
        "init": ("Dự đoán: He phù hợp với ReLU; Xavier có thể giữ phương sai cân bằng hơn giữa fan-in/fan-out.",
                 "Giải thích: so loss bước 0, độ lệch chuẩn kích hoạt và validation sau huấn luyện."),
    }
    lines += ["", "### Dự đoán trước, kết quả sau và giải thích"]
    for r in main_results:
        c, s = r["cfg"], r["summary"]
        if c["exp_id"] == "base-s1":
            continue
        delta = s["val_macro_f1"] - baseline_f1
        exceeds_noise = abs(delta) > 2 * sigma_f1 if len(seed_f1) > 1 else None
        prediction, mechanism = explanations.get(c["group"], ("Dự đoán trước: xem phần giả thuyết ở trên.", "Giải thích cơ chế theo số đo validation."))
        noise_text = ("chưa ước lượng do thiếu seed" if exceeds_noise is None else
                      ("vượt" if exceeds_noise else "không vượt") + " ngưỡng 2σ")
        lines += [f"", f"**{c['exp_id']} ({c['group']}).** {prediction}",
                  f"Quan sát: val macro-F1={s['val_macro_f1']:.4f}, chênh lệch so với base-s1={delta:+.4f}; {noise_text}. {mechanism}",
                  f"Biểu đồ: [thí nghiệm](figures/{c['exp_id']}.png)."]

    delta = final_result["summary"]["val_macro_f1"] - baseline_f1
    above = (abs(delta) > 2 * sigma_f1) if len(seed_f1) > 1 else False
    lines += ["", "## 4. Đánh giá cuối trên eval",
              "| cấu hình | val macro-F1 | eval macro-F1 | eval accuracy |",
              "|---|---:|---:|---:|",
              f"| Baseline (base-s1) | {baseline_f1:.4f} | {baseline_eval_scores['macro_f1']:.4f} | {baseline_eval_scores['accuracy']:.4f} |",
              f"| Cuối ({final_id}) | {final_result['summary']['val_macro_f1']:.4f} | {final_eval_scores['macro_f1']:.4f} | {final_eval_scores['accuracy']:.4f} |",
              f"- Chênh lệch val macro-F1 cuối so với baseline: {delta:+.4f}; vượt 2σ_seed: {above if len(seed_f1)>1 else 'chưa xác định (cần thêm seed)'}.",
              "", "### F1 theo lớp và phân tích nhầm lẫn",
              "| lớp | support | precision | recall | F1 |", "|---:|---:|---:|---:|---:|"]
    for cls in final_eval_scores["per_class"]:
        lines.append(f"| {cls['cls']} | {cls['support']} | {cls['precision']:.4f} | {cls['recall']:.4f} | {cls['f1']:.4f} |")
    cm = np.asarray(final_eval_scores["confusion_matrix"])
    hardest = min(final_eval_scores["per_class"], key=lambda row: row["f1"])
    confusion_row = cm[hardest["cls"]].copy()
    confusion_row[hardest["cls"]] = 0
    most_confused = int(confusion_row.argmax())
    lines += ["", f"Lớp F1 thấp nhất là **{hardest['cls']}** (F1={hardest['f1']:.4f}, support={hardest['support']}); trong các dự đoán sai, lớp bị nhầm nhiều nhất là **{most_confused}** ({int(confusion_row[most_confused])} mẫu).", "",
              "Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):",
              "| thật\dự đoán | " + " | ".join(str(i) for i in range(7)) + " |",
              "|---:|" + "---:|" * 7]
    for i, row in enumerate(cm):
        lines.append("| " + str(i) + " | " + " | ".join(str(int(value)) for value in row) + " |")
    lines += ["", "## 5. Câu hỏi dẫn dắt và hạn chế",
              "- Dropout: dùng kết quả train-val gap và validation macro-F1 phía trên để kết luận có dấu hiệu overfit hay không.",
              "- Clipping: xem grad norm trước clip; thí nghiệm này dùng learning rate baseline, không khẳng định xử lý được mọi trường hợp gradient nổ.",
              "- Mixed precision: kết luận nhanh/chậm chỉ dựa trên số đo của runtime này; T4/GPU khác có thể cho kết quả khác.",
              "- Nếu loss không giảm: (1) kiểm tra nhãn/dữ liệu, chuẩn hoá và loss bước 0; (2) thử overfit 20 mẫu; (3) kiểm tra gradient từng tham số và grad_norm, sau đó rà soát learning rate/zero_grad.",
              "- Hạn chế: hai seed baseline cho ước lượng nhiễu còn thô; learning-rate pilots chỉ 4 epoch; kết quả chỉ đại diện cho phần cứng Colab đã chạy. Mọi lựa chọn cấu hình dựa trên validation; eval chỉ được chấm sau khi chốt.", ""]
    report_path = OUT_DIR / "REPORT.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print("Đã lưu báo cáo:", report_path)

write_measured_report()
baseline_csv.unlink(missing_ok=True)
baseline_json.unlink(missing_ok=True)
zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR.parent,
                               base_dir=OUT_DIR.name)
print("Tệp submission:", sorted(path.name for path in OUT_DIR.iterdir()))
print("Gói nộp:", zip_path)
